# Single Inheritance: An Advanced, Step-by-Step Problem-Solving Tutorial

**A completely new problem set with worked solutions.** This notebook follows the *teaching rhythm* of your supplied lesson: introduce a relationship, ask a question, run a small experiment, explain its result, then add exactly one more idea. We avoid the pattern of dumping a full solution into one code cell.

**The source lesson's starting point:** `Shape → Ellipse → Circle` and `Shape → Polygon → Rectangle → Square`, plus `Triangle` under `Polygon`. It also introduces `issubclass`, `isinstance`, indirect subclassing, and the universal `object` ancestor. Those are the foundations; all applications below go beyond the supplied introductory material.

**What is new here:** 15 worked problems built around dispatch order, attribute lookup, method binding, constructors, immutable geometry, shared versus instance state, special methods, abstract contracts, properties, inherited factories, equality, runtime-built subclasses, inheritance versus composition, a processing-system capstone, and source-of-method diagnostics.

**How to use:** Run cells *in order* with Python 3.10+; the Python standard library is enough. For each **Predict** prompt, pause and decide before running the immediately following code. Every expected failure is caught deliberately. Assertions serve as self-checks. Each problem introduces uniquely named classes, so running the entire notebook top-to-bottom does not depend on restarting between problems.

## Reading the tutorial

The notation `A → B → C` means `B(A)` and `C(B)`; arrows point toward a descendant. A **direct** parent is in `__bases__`, whereas an **indirect** ancestor appears farther along `__mro__`. `issubclass` concerns classes; `isinstance` concerns instances.

A solution is spread across several adjacent cells. An explanation after a code cell interprets what just happened, rather than assuming the output speaks for itself. At the end of each problem, an assertion cell confirms the intended contract.

---

## Problem 01 — The handler-order trap

**Challenge.** Write a shape classifier that recognizes the most specialized type rather than prematurely returning an ancestor category.

**Before running anything:** Will testing `Shape` first ever allow a `Circle` to reach a later `Circle` branch?

### Step 1 — Reconstruct the small hierarchy

We start with the relationships in the supplied lesson, without adding behavior yet.

In [1]:
class TShape:
    pass

class TEllipse(TShape):
    pass

class TCircle(TEllipse):
    pass

class TPolygon(TShape):
    pass

class TRectangle(TPolygon):
    pass

class TSquare(TRectangle):
    pass

class TTriangle(TPolygon):
    pass

### Step 2 — Observe the broad match

`TCircle` is indirectly a subclass of `TShape`; therefore the first condition below will match before Python can examine any later branch.

In [2]:
def classify_bad(item):
    if isinstance(item, TShape):
        return 'shape'
    if isinstance(item, TCircle):
        return 'circle'
    return 'unknown'

print(classify_bad(TCircle()))

shape


**Explanation.** The output is `shape`, not `circle`. An `if`/`return` decision cannot get to a more specific check after the broad test has already succeeded. This is about *control-flow order*, not a failure in `isinstance`.

### Step 3 — Put descendants before ancestors

A local repair is to order checks from more specific to more general. This is appropriate when you really must classify by runtime type.

In [3]:
def classify_shape(item):
    if isinstance(item, TSquare):
        return 'square'
    if isinstance(item, TRectangle):
        return 'rectangle'
    if isinstance(item, TCircle):
        return 'circle'
    if isinstance(item, TEllipse):
        return 'ellipse'
    if isinstance(item, TTriangle):
        return 'triangle'
    if isinstance(item, TPolygon):
        return 'polygon'
    if isinstance(item, TShape):
        return 'shape'
    return 'unknown'

### Step 4 — Test both branches

A square must beat the broader rectangle test, and a circle must beat the ellipse test.

In [4]:
examples = [TCircle(), TEllipse(), TSquare(), TRectangle(), TTriangle(), TPolygon(), TShape(), object()]
results = [classify_shape(item) for item in examples]
print(results)
assert results == ['circle', 'ellipse', 'square', 'rectangle', 'triangle', 'polygon', 'shape', 'unknown']

['circle', 'ellipse', 'square', 'rectangle', 'triangle', 'polygon', 'shape', 'unknown']


### What we have proved

Ancestor checks are *inclusive*. Descendant-first checks repair classification, but often a better design is to **ask the object to perform behavior** rather than identify its type. We will implement that idea in later problems.

---

## Problem 02 — Why inherited methods become bound

**Challenge.** Explain precisely where a method is defined, how it is found, and how `self` is supplied when it is called through an instance.

**Before running anything:** Does a subclass copy each inherited function into its own `__dict__`?

### Step 1 — Define one method, then inherit it

`TAlert` does not define `label`; `TCriticalAlert` inherits that same implementation too.

In [5]:
class TSignal:
    def label(self):
        return f'signal:{type(self).__name__}'

class TAlert(TSignal):
    pass

class TCriticalAlert(TAlert):
    pass

alert = TCriticalAlert()
print(alert.label())

signal:TCriticalAlert


### Step 2 — Look at each class dictionary

A class dictionary only records names *defined on that class*, while normal attribute access walks the inheritance lookup path.

In [6]:
print('label in critical:', 'label' in TCriticalAlert.__dict__)
print('label in alert:', 'label' in TAlert.__dict__)
print('label in signal:', 'label' in TSignal.__dict__)
print('MRO:', [cls.__name__ for cls in TCriticalAlert.__mro__])

label in critical: False
label in alert: False
label in signal: True
MRO: ['TCriticalAlert', 'TAlert', 'TSignal', 'object']


**Explanation.** The implementation resides only in `TSignal.__dict__`. The subclass does not need to duplicate it. `__mro__` shows the normal class lookup order for this ordinary single-inheritance chain.

### Step 3 — Compare the function with the bound method

Reading the function from the class and reading it from an instance yield related but different objects.

In [7]:
raw_function = TSignal.__dict__['label']
bound_method = alert.label
print('bound instance:', bound_method.__self__ is alert)
print('underlying function:', bound_method.__func__ is raw_function)
print('manual call:', raw_function(alert))
print('usual call:', bound_method())

bound instance: True
underlying function: True
manual call: signal:TCriticalAlert
usual call: signal:TCriticalAlert


**Explanation.** Python binds `alert` as the method receiver. The raw function can be invoked explicitly with that same receiver. `type(self)` remains `TCriticalAlert`, even when the function was written on `TSignal`.

### Step 4 — Compare inherited lookup with an override

Adding a `label` to the middle class changes the method used by the leaf, without changing the leaf definition.

In [8]:
class TAlert(TSignal):
    def label(self):
        return f'alert -> {super().label()}'

class TCriticalAlert(TAlert):
    pass

print(TCriticalAlert().label())
assert TCriticalAlert().label() == 'alert -> signal:TCriticalAlert'

alert -> signal:TCriticalAlert


**Explanation.** The new class definitions replace the old names in this notebook. The *new* `TCriticalAlert` inherits from the *new* `TAlert`. Its inherited method calls `super()` to preserve the ancestor behavior.

### What we have proved

A class normally stores only definitions made at that level. Inherited functions are located through lookup and become bound methods when accessed on an instance. Redefining a class name creates a new class; it does not retroactively rewire existing instances of an old class.

---

## Problem 03 — Repair a broken initialization chain

**Challenge.** Diagnose a subclass that leaves required parent state uninitialized, then build a verified three-stage initialization chain.

**Before running anything:** Can an inherited method work when a subclass skipped the parent `__init__`?

### Step 1 — Give the parent a real invariant

Every ticket must have a nonempty reference. `summary()` relies on it.

In [9]:
class TTicket:
    def __init__(self, reference):
        if not isinstance(reference, str) or not reference.strip():
            raise ValueError('reference must be nonempty')
        self.reference = reference.strip()

    def summary(self):
        return f'ticket:{self.reference}'

### Step 2 — See the defective descendant

This descendant forgets to call `super().__init__`. Its constructor finishes, but inherited functionality breaks later.

In [10]:
class TBrokenPriorityTicket(TTicket):
    def __init__(self, reference, priority):
        self.priority = priority

broken = TBrokenPriorityTicket('A-17', 2)
try:
    print(broken.summary())
except AttributeError as error:
    print(type(error).__name__, str(error))

AttributeError 'TBrokenPriorityTicket' object has no attribute 'reference'


**Explanation.** The missing `reference` is not an inheritance lookup problem: it is an **instance state** problem. The parent constructor did not run, so the parent invariant was never established.

### Step 3 — Delegate parent state, then add subclass state

Validate each level at the level responsible for that piece of information.

In [11]:
class TPriorityTicket(TTicket):
    def __init__(self, reference, priority):
        super().__init__(reference)
        if type(priority) is not int or priority not in range(1, 6):
            raise ValueError('priority must be an integer from 1 to 5')
        self.priority = priority

    def summary(self):
        return f'{super().summary()} priority:{self.priority}'

### Step 4 — Introduce a grandchild that delegates again

Each subclass extends the previous level, without reimplementing its validation.

In [12]:
class TAssignedTicket(TPriorityTicket):
    def __init__(self, reference, priority, assignee):
        super().__init__(reference, priority)
        if not isinstance(assignee, str) or not assignee.strip():
            raise ValueError('assignee must be nonempty')
        self.assignee = assignee.strip()

    def summary(self):
        return f'{super().summary()} assignee:{self.assignee}'

print(TAssignedTicket(' A-17 ', 2, ' Ana ').summary())

ticket:A-17 priority:2 assignee:Ana


### Step 5 — Test successful and failed construction

Make validation failures explicit; a boolean must not pass as an integer priority merely because `bool` subclasses `int`.

In [13]:
ticket = TAssignedTicket(' A-17 ', 2, ' Ana ')
assert ticket.summary() == 'ticket:A-17 priority:2 assignee:Ana'
for bad_priority in (True, 0, 6, 2.5):
    try:
        TPriorityTicket('X', bad_priority)
    except ValueError:
        print('rejected:', repr(bad_priority))
    else:
        raise AssertionError(f'accepted invalid priority: {bad_priority!r}')

rejected: True
rejected: 0
rejected: 6
rejected: 2.5


### What we have proved

The parent owns the `reference` invariant, the child owns `priority`, and the grandchild owns `assignee`. An inherited method is safe only if all state on which it depends is properly initialized.

---

## Problem 04 — Make the Circle–Ellipse relationship operational

**Challenge.** Turn the lesson’s empty `Ellipse → Circle` chain into a defensible *immutable* geometry model; use indirect inheritance to reuse area and eccentricity calculations.

**Before running anything:** If circle radii could be independently changed, would it remain a circle?

### Step 1 — Introduce private radii and read-only properties

We use a simple convention: regular attribute assignment through the public API cannot change dimensions after construction. This is not a security/absolute-immutability guarantee.

In [14]:
import math

class TEllipseGeometry:
    def __init__(self, horizontal_radius, vertical_radius):
        for value in (horizontal_radius, vertical_radius):
            if isinstance(value, bool) or not isinstance(value, (int, float)):
                raise TypeError('radii must be real numbers, not booleans')
            if not math.isfinite(value) or value <= 0:
                raise ValueError('radii must be finite and positive')
        self._horizontal_radius = float(horizontal_radius)
        self._vertical_radius = float(vertical_radius)

    @property
    def horizontal_radius(self):
        return self._horizontal_radius

    @property
    def vertical_radius(self):
        return self._vertical_radius

### Step 2 — Put genuinely shared behavior on the parent

The area formula is valid for any ellipse, including the special case of a circle.

In [15]:
def ellipse_area(self):
    return math.pi * self.horizontal_radius * self.vertical_radius

TEllipseGeometry.area = ellipse_area
print(round(TEllipseGeometry(2, 3).area(), 5))

18.84956


**Explanation.** Assigning a function to the class here demonstrates that classes can be extended dynamically for exploration. For ordinary application code, define the method inside the class instead; we will produce the final clean definitions below.

### Step 3 — Write the clean parent and child

The circle supplies the *same* radius twice to its ellipse parent. The parent’s validated state and area implementation can be reused unchanged.

In [16]:
class TEllipseGeometry:
    def __init__(self, horizontal_radius, vertical_radius):
        for radius in (horizontal_radius, vertical_radius):
            if isinstance(radius, bool) or not isinstance(radius, (int, float)):
                raise TypeError('radii must be real numbers, not booleans')
            if not math.isfinite(radius) or radius <= 0:
                raise ValueError('radii must be finite and positive')
        self._horizontal_radius = float(horizontal_radius)
        self._vertical_radius = float(vertical_radius)

    @property
    def horizontal_radius(self):
        return self._horizontal_radius

    @property
    def vertical_radius(self):
        return self._vertical_radius

    def area(self):
        return math.pi * self.horizontal_radius * self.vertical_radius

class TCircleGeometry(TEllipseGeometry):
    def __init__(self, radius):
        super().__init__(radius, radius)

    @property
    def radius(self):
        return self.horizontal_radius

### Step 4 — Prove that circle behavior really is inherited

Inspect the direct parent, the absence of a circle-specific `area`, and the computed areas.

In [17]:
circle = TCircleGeometry(3)
ellipse = TEllipseGeometry(2, 4)
print(TCircleGeometry.__bases__)
print('circle defines area?', 'area' in TCircleGeometry.__dict__)
print('circle area:', round(circle.area(), 5))
print('ellipse area:', round(ellipse.area(), 5))

(<class '__main__.TEllipseGeometry'>,)
circle defines area? False
circle area: 28.27433
ellipse area: 25.13274


### Step 5 — Test the invariant and reject impossible inputs

A normal assignment to either public radius property is refused because those properties have no setters.

In [18]:
assert math.isclose(circle.area(), 9 * math.pi)
assert circle.horizontal_radius == circle.vertical_radius == circle.radius == 3
assert isinstance(circle, TEllipseGeometry)
try:
    circle.horizontal_radius = 99
except AttributeError:
    print('public radius is read-only')
else:
    raise AssertionError('read-only property unexpectedly accepted assignment')
for bad in (0, -1, float('nan'), float('inf'), True):
    try:
        TCircleGeometry(bad)
    except (TypeError, ValueError):
        print('rejected radius:', bad)
    else:
        raise AssertionError('accepted invalid radius')

public radius is read-only
rejected radius: 0
rejected radius: -1
rejected radius: nan
rejected radius: inf
rejected radius: True


### What we have proved

This is a sensible instance of single inheritance because `TCircleGeometry` can honor its inherited, read-only ellipse API. Private underscores and read-only properties establish a normal public convention, not absolute protection against deliberate internal mutation.

---

## Problem 05 — Extend behavior without losing the audit trail

**Challenge.** Implement a three-level notification family in which each override adds a single processing entry and retains what the parent did.

**Before running anything:** What goes missing if an overriding method never delegates to its parent?

### Step 1 — Define the shared operation

A base notification adds a generic status to a passed-in log and returns that same log.

In [19]:
class TNotice:
    def deliver(self, log):
        log.append('notice prepared')
        return log

log = []
assert TNotice().deliver(log) is log
print(log)

['notice prepared']


### Step 2 — Make a broken override visible

Returning a new list discards a parent operation and potentially breaks callers relying on object identity.

In [20]:
class TBrokenEmailNotice(TNotice):
    def deliver(self, log):
        return ['email sent']

log = []
result = TBrokenEmailNotice().deliver(log)
print('caller log:', log, '| returned log:', result)

caller log: [] | returned log: ['email sent']


**Explanation.** The subclass changed the return-object contract. It also silently skipped the base audit step. We should preserve the parent behavior before appending the specialization.

### Step 3 — Use `super()` and keep one list

In a single-inheritance chain, each override contributes its own part to the same operation.

In [21]:
class TEmailNotice(TNotice):
    def deliver(self, log):
        result = super().deliver(log)
        result.append('email sent')
        return result

class TUrgentEmailNotice(TEmailNotice):
    def deliver(self, log):
        result = super().deliver(log)
        result.append('urgent receipt requested')
        return result

### Step 4 — Inspect the call result and sequence

The grandchild delegates first, so messages appear in ancestor-to-descendant order.

In [22]:
audit = []
returned = TUrgentEmailNotice().deliver(audit)
print(audit)
assert returned is audit
assert audit == ['notice prepared', 'email sent', 'urgent receipt requested']

['notice prepared', 'email sent', 'urgent receipt requested']


**Explanation.** In this example, delegation is intentional: each level extends the parent’s contract. There are also legitimate cases where an override *replaces* parent behavior, but then the replacement must still satisfy the caller’s documented contract.

### What we have proved

Use `super()` when the derived behavior builds on the base behavior. Verify both values and side effects, especially if methods mutate shared objects passed by the caller.

---

## Problem 06 — Separate class defaults from instance overrides

**Challenge.** Explore attribute lookup when a base class provides a configuration default, a subclass changes that default, and individual instances override it.

**Before running anything:** If one instance assigns `self.format`, does it change the class default for every other instance?

### Step 1 — Create three levels of defaults

The derived classes change a class-level setting; they do not redeclare the whole API.

In [23]:
class TFormatter:
    format = 'plain'

    def describe(self):
        return f'{type(self).__name__}:{self.format}'

class THtmlFormatter(TFormatter):
    format = 'html'

class TCompactHtmlFormatter(THtmlFormatter):
    pass

print(TCompactHtmlFormatter().describe())

TCompactHtmlFormatter:html


### Step 2 — Change one instance only

Attribute lookup normally considers an instance’s own dictionary before an ordinary class variable.

In [24]:
first = TCompactHtmlFormatter()
second = TCompactHtmlFormatter()
first.format = 'mini-html'
print('first:', first.describe())
print('second:', second.describe())
print('instance dictionary:', first.__dict__)

first: TCompactHtmlFormatter:mini-html
second: TCompactHtmlFormatter:html
instance dictionary: {'format': 'mini-html'}


### Step 3 — Look up where each name lives

The leaf does not have its own class-level `format`; the middle class does.

In [25]:
print('leaf local format:', TCompactHtmlFormatter.__dict__.get('format', '<not defined>'))
print('middle local format:', THtmlFormatter.__dict__['format'])
print('base local format:', TFormatter.__dict__['format'])

leaf local format: <not defined>
middle local format: html
base local format: plain


**Explanation.** An ordinary class attribute is shared through lookup until a more specific class or an instance shadows it. Unlike a mutable shared list, immutable default strings are relatively safe class attributes.

### Step 4 — Demonstrate and repair a mutable class-attribute trap

A list on the class is shared by all instances unless each object initializes its own list.

In [26]:
class TBadHistory:
    entries = []

one, two = TBadHistory(), TBadHistory()
one.entries.append('first')
print('two unexpectedly sees:', two.entries)

class TGoodHistory:
    def __init__(self):
        self.entries = []

one, two = TGoodHistory(), TGoodHistory()
one.entries.append('first')
print('two now sees:', two.entries)
assert two.entries == []

two unexpectedly sees: ['first']
two now sees: []


### What we have proved

Class attributes are appropriate for shared constants and defaults. Initialize mutable, per-object data in `__init__` instead. Assigning to `self.name` usually creates an instance attribute; mutating a shared class-level object does not.

---

## Problem 07 — Discover why Python ignores an instance-level `__len__`

**Challenge.** Implement the container protocol in a single-inheritance chain and show that implicit special-method lookup differs from ordinary attribute access.

**Before running anything:** If we set `obj.__len__ = lambda: 100`, will `len(obj)` become 100?

### Step 1 — Implement a reusable container base

`len` is defined at the class level, and the parent owns the stored items.

In [27]:
class TCollection:
    def __init__(self, items):
        self._items = tuple(items)

    def __len__(self):
        return len(self._items)

    def __iter__(self):
        return iter(self._items)

collection = TCollection([1, 2, 3])
print(len(collection), list(collection))

3 [1, 2, 3]


### Step 2 — Narrow visible items in a subclass

The subclass introduces a virtual `visible_items` view and deliberately overrides both relevant operations so their results stay consistent.

In [28]:
class TPositiveCollection(TCollection):
    @property
    def visible_items(self):
        return tuple(value for value in self._items if value > 0)

    def __len__(self):
        return len(self.visible_items)

    def __iter__(self):
        return iter(self.visible_items)

positive = TPositiveCollection([-2, 1, 0, 5])
print('length:', len(positive), '| items:', list(positive))

length: 2 | items: [1, 5]


### Step 3 — Try an instance-level special-method replacement

Ordinary dotted attribute access finds the instance attribute, but the built-in `len` uses a special lookup on the type.

In [29]:
positive.__len__ = lambda: 100
print('explicit attribute call:', positive.__len__())
print('implicit len call:', len(positive))
assert positive.__len__() == 100
assert len(positive) == 2

explicit attribute call: 100
implicit len call: 2


**Explanation.** The special-method behavior is defined by the class hierarchy. Overwriting the instance attribute is **not** a reliable way to customize Python syntax or built-ins such as `len`, `iter`, arithmetic operators, and so on. Overriding the method on the *subclass* is the intended route.

### Step 4 — Prove the grandchild inherits the override

No new `__len__` implementation is needed at this level.

In [30]:
class TNamedPositiveCollection(TPositiveCollection):
    def __init__(self, name, items):
        super().__init__(items)
        self.name = name

named = TNamedPositiveCollection('measurements', [-1, 4, 7])
print(named.name, len(named), list(named))
assert '__len__' not in TNamedPositiveCollection.__dict__
assert len(named) == 2

measurements 2 [4, 7]


### What we have proved

Python’s implicit special methods are normally resolved on the class, not through an arbitrary instance attribute of the same name. A subclass override participates in that lookup and is inherited by its descendants.

---

## Problem 08 — Distinguish documentation-only interfaces from enforced abstract contracts

**Challenge.** Compare a parent that merely raises `NotImplementedError` with an abstract base that prevents instantiation until the method is implemented.

**Before running anything:** When does Python discover an incomplete child: on construction or only on method call?

### Step 1 — Demonstrate a conventional, non-enforced stub

Nothing prevents constructing this class. The failure happens only if the stub method is called.

In [31]:
class TRunnableStub:
    def run(self):
        raise NotImplementedError('subclass must implement run')

stub = TRunnableStub()
print('constructed:', type(stub).__name__)
try:
    stub.run()
except NotImplementedError as error:
    print('failed on call:', error)

constructed: TRunnableStub
failed on call: subclass must implement run


### Step 2 — Define a true abstract method

The `abc` module makes the contract explicit and refuses incomplete instances.

In [32]:
from abc import ABC, abstractmethod

class TWorkflow(ABC):
    @abstractmethod
    def run(self):
        raise NotImplementedError('abstract: implement run in a concrete child')

try:
    TWorkflow()
except TypeError as error:
    print('failed on construction:', type(error).__name__)

failed on construction: TypeError


### Step 3 — Implement a concrete descendant

A concrete implementation makes the descendant instantiable. It also creates behavior that later descendants can inherit.

In [33]:
class TCountingWorkflow(TWorkflow):
    def __init__(self, values):
        self.values = tuple(values)

    def run(self):
        return {'count': len(self.values)}

print(TCountingWorkflow([4, 5]).run())

{'count': 2}


### Step 4 — Extend a concrete descendant without reimplementing the abstract method

The grandchild is concrete because the parent already satisfies the contract.

In [34]:
class TNamedCountingWorkflow(TCountingWorkflow):
    def __init__(self, name, values):
        super().__init__(values)
        self.name = name

    def run(self):
        result = super().run()
        return {'name': self.name, **result}

named_workflow = TNamedCountingWorkflow('batch-1', [2, 3, 4])
print(named_workflow.run())
assert isinstance(named_workflow, TWorkflow)
assert named_workflow.run() == {'name': 'batch-1', 'count': 3}

{'name': 'batch-1', 'count': 3}


**Explanation.** `ABC` is itself a base class. `TCountingWorkflow( TWorkflow )` and its descendant each have exactly one explicit base in this example; there is no multiple inheritance in this problem.

### What we have proved

An ordinary `NotImplementedError` stub is a convention that fails when called; `@abstractmethod` provides construction-time enforcement. A descendant need not repeat an abstract method once a concrete ancestor implements it.

---

## Problem 09 — Understand why a property beats instance-dictionary shadowing

**Challenge.** Compare ordinary class-variable shadowing with read-only property behavior, then enforce a validated value across a subclass chain.

**Before running anything:** Would `obj.__dict__["level"] = -5` override a parent `@property level`?

### Step 1 — Create a validated parent property

The backing attribute is `_level`; the public name is mediated by a property with a setter.

In [35]:
class TLevel:
    def __init__(self, level):
        self.level = level

    @property
    def level(self):
        return self._level

    @level.setter
    def level(self, value):
        if type(value) is not int or not 0 <= value <= 100:
            raise ValueError('level must be an integer from 0 to 100')
        self._level = value

reading = TLevel(40)
print(reading.level)

40


### Step 2 — Try to fool the getter with an instance-dictionary key

A property with setter behavior is a *data descriptor*. It takes precedence over the instance dictionary for the same public name.

In [36]:
reading.__dict__['level'] = -5
print('dictionary value:', reading.__dict__['level'])
print('property value:', reading.level)
assert reading.level == 40

dictionary value: -5
property value: 40


**Explanation.** A manual edit to `__dict__` can store an irrelevant entry, but normal property access still consults the descriptor. Do not treat this as a security guarantee: internal attributes can still be deliberately modified.

### Step 3 — Inherit the validation unchanged

The subclass adds a label and reads the same validated property; it does not redefine level handling.

In [37]:
class TLabeledLevel(TLevel):
    def __init__(self, level, label):
        super().__init__(level)
        if not isinstance(label, str) or not label.strip():
            raise ValueError('label must be nonempty')
        self.label = label.strip()

    def display(self):
        return f'{self.label}: {self.level}'

labeled = TLabeledLevel(75, 'CPU')
print(labeled.display())

CPU: 75


### Step 4 — Exercise both construction-time and mutation-time validation

One common mistake is to validate only in `__init__`, leaving later assignments unchecked.

In [38]:
labeled.level = 80
assert labeled.display() == 'CPU: 80'
for invalid in (-1, 101, True, 4.2):
    try:
        labeled.level = invalid
    except ValueError:
        print('rejected:', repr(invalid))
    else:
        raise AssertionError('invalid assignment unexpectedly succeeded')
assert labeled.level == 80

rejected: -1
rejected: 101
rejected: True
rejected: 4.2


### What we have proved

A property is inherited like other class attributes, but as a data descriptor it controls normal public access more strongly than an ordinary default variable. Validate through the property at construction and assignment.

---

## Problem 10 — Build an inherited parser that returns the requested subclass

**Challenge.** Use `@classmethod` plus `cls` so `parse` can construct either the parent or a descendant. Examine how a subclass can change its parsing requirements safely.

**Before running anything:** Does calling `Child.parse(...)` necessarily construct a `Child` even when `parse` is defined only on `Parent`?

### Step 1 — Define a shared, strict two-coordinate parser

The classmethod will delegate construction to `cls`, while a normal static parsing helper only extracts numbers.

In [39]:
class TPoint:
    def __init__(self, x, y):
        if any(isinstance(v, bool) or not isinstance(v, (int, float)) or not math.isfinite(v)
               for v in (x, y)):
            raise ValueError('coordinates must be finite numbers')
        self.x = float(x)
        self.y = float(y)

    @classmethod
    def parse(cls, text, **kwargs):
        if not isinstance(text, str):
            raise TypeError('point input must be text')
        parts = text.split(',')
        if len(parts) != 2:
            raise ValueError('expected exactly x,y')
        try:
            x, y = (float(part.strip()) for part in parts)
        except ValueError as error:
            raise ValueError('invalid coordinate text') from error
        return cls(x, y, **kwargs)

### Step 2 — Prove the base parser

Notice that the helper does not need to know any concrete subclass yet.

In [40]:
point = TPoint.parse('3, 4')
print(type(point).__name__, point.x, point.y)
assert type(point) is TPoint

TPoint 3.0 4.0


### Step 3 — Add a subclass requiring an extra argument

An inherited factory forwards explicitly documented subclass options through keyword arguments.

In [41]:
class TLabeledPoint(TPoint):
    def __init__(self, x, y, *, label):
        super().__init__(x, y)
        if not isinstance(label, str) or not label.strip():
            raise ValueError('label must be nonempty')
        self.label = label.strip()

labeled_point = TLabeledPoint.parse('5,6', label='origin-ish')
print(type(labeled_point).__name__, labeled_point.label)
assert type(labeled_point) is TLabeledPoint

TLabeledPoint origin-ish


### Step 4 — Document what happens if subclass arguments are omitted

Inheritance does not magically know which extra constructor arguments the subclass needs; normal Python argument checking still applies.

In [42]:
try:
    TLabeledPoint.parse('5,6')
except TypeError as error:
    print('missing subclass-specific input:', type(error).__name__)
else:
    raise AssertionError('required label unexpectedly optional')

missing subclass-specific input: TypeError


### Step 5 — Test malformed text and preserved types

We do not silently ignore extra coordinates or accept non-finite numbers.

In [43]:
for text in ('1', '1,2,3', 'a,2', 'nan,2'):
    try:
        TPoint.parse(text)
    except ValueError:
        print('rejected:', repr(text))
    else:
        raise AssertionError('invalid coordinates accepted')
assert issubclass(TLabeledPoint, TPoint)
assert isinstance(labeled_point, TPoint)

rejected: '1'
rejected: '1,2,3'
rejected: 'a,2'
rejected: 'nan,2'


### What we have proved

An inherited classmethod receives the class used for invocation through `cls`. `cls(...)` preserves the requested type, provided the parsed data and forwarded options satisfy that subclass’s constructor contract.

---

## Problem 11 — Write equality rules that remain symmetric across a hierarchy

**Challenge.** Examine the difference between compatible type checks and equality’s exact-type policies. Avoid silently erasing subclass-specific state.

**Before running anything:** Should two objects with the same base identifier compare equal when one has an additional version field?

### Step 1 — Create a base class with an explicit policy

Our chosen policy: records compare by identifier **only when they have exactly the same concrete type**. Other types get `NotImplemented`, allowing Python to try the other operand.

In [44]:
class TRecord:
    def __init__(self, identifier):
        if not isinstance(identifier, str) or not identifier:
            raise ValueError('identifier must be nonempty text')
        self.identifier = identifier

    def __eq__(self, other):
        if type(self) is not type(other):
            return NotImplemented
        return self.identifier == other.identifier

    __hash__ = None

### Step 2 — Extend the class with version state

The grandchild needs to account for new state in comparisons, so its override first compares the shared identifier and then its own version.

In [45]:
class TVersionedRecord(TRecord):
    def __init__(self, identifier, version):
        super().__init__(identifier)
        if type(version) is not int or version < 1:
            raise ValueError('version must be a positive integer')
        self.version = version

    def __eq__(self, other):
        if type(self) is not type(other):
            return NotImplemented
        return super().__eq__(other) and self.version == other.version

    __hash__ = None

### Step 3 — Compare in both directions

The two types share an ancestor, but the chosen equality policy keeps their values distinct.

In [46]:
base = TRecord('r-1')
version_a = TVersionedRecord('r-1', 1)
version_b = TVersionedRecord('r-1', 2)
version_a_copy = TVersionedRecord('r-1', 1)
print(base == version_a, version_a == base)
print(version_a == version_b, version_a == version_a_copy)
assert base != version_a and version_a != base
assert version_a != version_b and version_a == version_a_copy

False False
False True


### Step 4 — Test hash and demonstrate a third level

Overriding `__eq__` means mutable value objects should normally remain unhashable unless a compatible immutable hashing policy is explicitly designed.

In [47]:
class TTaggedVersionedRecord(TVersionedRecord):
    def __init__(self, identifier, version, tag):
        super().__init__(identifier, version)
        self.tag = tag

    def __eq__(self, other):
        if type(self) is not type(other):
            return NotImplemented
        return super().__eq__(other) and self.tag == other.tag

    __hash__ = None

assert TTaggedVersionedRecord('r-1', 1, 'a') != TTaggedVersionedRecord('r-1', 1, 'b')
try:
    hash(version_a)
except TypeError:
    print('mutable value object is unhashable')

mutable value object is unhashable


**Explanation.** Python may offer the reflected comparison first when the right-hand type is a subclass; checking **both** orders ensures our policy is symmetric. Here subclass implementations compare shared state via `super().__eq__` only after exact type matches.

### What we have proved

Inheritance alone does not define appropriate equality. Decide whether cross-type values are equal, account for subclass fields, return `NotImplemented` for unsupported comparisons, and ensure hashing is compatible with equality.

---

## Problem 12 — Create subclasses dynamically without introducing multiple inheritance

**Challenge.** Use the class-building form of `type` to construct a single-inheritance chain at runtime. Check names, bases, instances, and inherited methods.

**Before running anything:** Is a dynamically created class any less of a class than one created with `class` syntax?

### Step 1 — Start with an ordinary base

This is a deliberately small parent with a stable instance API.

In [48]:
class TRuntimeBase:
    category = 'base'

    def describe(self):
        return f'{type(self).__name__}:{self.category}'

print(TRuntimeBase().describe())

TRuntimeBase:base


### Step 2 — Build one direct child with `type`

The three-argument form takes a class name, a *tuple of bases*, and a namespace dictionary.

In [49]:
TRuntimeChild = type('TRuntimeChild', (TRuntimeBase,), {'category': 'dynamic'})
child = TRuntimeChild()
print(child.describe())
print('bases:', TRuntimeChild.__bases__)
print('is subclass:', issubclass(TRuntimeChild, TRuntimeBase))

TRuntimeChild:dynamic
bases: (<class '__main__.TRuntimeBase'>,)
is subclass: True


### Step 3 — Build a grandchild and add behavior

A function in the namespace is bound as a method on instances, just as a `def` in a class body would be.

In [50]:
def runtime_label(self):
    return self.describe().upper()

TRuntimeGrandchild = type(
    'TRuntimeGrandchild',
    (TRuntimeChild,),
    {'category': 'grandchild', 'label': runtime_label},
)
grandchild = TRuntimeGrandchild()
print(grandchild.label())
print('MRO:', [cls.__name__ for cls in TRuntimeGrandchild.__mro__])

TRUNTIMEGRANDCHILD:GRANDCHILD
MRO: ['TRuntimeGrandchild', 'TRuntimeChild', 'TRuntimeBase', 'object']


### Step 4 — Check class versus instance questions

`type` builds the class; `isinstance` checks an object created from it. The ordinary inheritance semantics are unchanged.

In [51]:
assert isinstance(grandchild, TRuntimeBase)
assert issubclass(TRuntimeGrandchild, TRuntimeChild)
assert TRuntimeGrandchild.__bases__ == (TRuntimeChild,)
assert grandchild.label() == 'TRUNTIMEGRANDCHILD:GRANDCHILD'
print('all runtime-class assertions passed')

all runtime-class assertions passed


**Explanation.** Dynamic class construction is useful for metaprogramming and certain frameworks, but it increases the burden of reading and debugging code. Prefer normal `class` definitions unless runtime construction serves a concrete requirement.

### What we have proved

A class created with `type(name, (one_base,), namespace)` obeys the same class/instance distinctions and transitive inheritance rules as a class written using the `class` statement.

---

## Problem 13 — Choose composition when the proposed child is not really a subtype

**Challenge.** Diagnose a cache wrapper that changes what callers mean by “storage.” Implement the wrapper using a **has-a** relationship while still allowing a separate, valid inheritance hierarchy.

**Before running anything:** Does “uses a store” imply “is a store”?

### Step 1 — Define the storage contract

The underlying store supports writing bytes and reading them back by key. A subclass should preserve this contract.

In [52]:
class TMemoryStore:
    def __init__(self):
        self._values = {}

    def put(self, key, data):
        if not isinstance(key, str) or not key:
            raise ValueError('key must be nonempty text')
        if not isinstance(data, bytes):
            raise TypeError('data must be bytes')
        self._values[key] = data

    def get(self, key):
        return self._values[key]

store = TMemoryStore()
store.put('a', b'example')
print(store.get('a'))

b'example'


### Step 2 — Show the tempting, misleading subclass

This object is called a cache wrapper but adds an entirely different `get` contract: returning a fallback for missing keys. Callers expecting `KeyError` might be surprised.

In [53]:
class TSurprisingStore(TMemoryStore):
    def get(self, key):
        try:
            return super().get(key)
        except KeyError:
            return b''

surprise = TSurprisingStore()
print('missing key becomes:', surprise.get('missing'))

missing key becomes: b''


**Explanation.** The example demonstrates a potential contract violation **when** callers rely on `get` raising `KeyError` for a missing key. A concrete inheritance decision depends on the documented parent contract, not just matching method names.

### Step 3 — Use composition for an optional fallback view

The wrapper *contains* a store. It offers a deliberately different method name and no misleading `isinstance(wrapper, TMemoryStore)` relationship.

In [54]:
class TFallbackView:
    def __init__(self, store, fallback=b''):
        if not isinstance(store, TMemoryStore):
            raise TypeError('store must be a TMemoryStore')
        if not isinstance(fallback, bytes):
            raise TypeError('fallback must be bytes')
        self.store = store
        self.fallback = fallback

    def get_or_default(self, key):
        try:
            return self.store.get(key)
        except KeyError:
            return self.fallback

view = TFallbackView(store, fallback=b'none')
print(view.get_or_default('a'), view.get_or_default('missing'))

b'example' b'none'


### Step 4 — Add a genuine specialized store

A logging store can preserve the original `get` behavior and extend it with a side effect; this is a more defensible subclass.

In [55]:
class TLoggingStore(TMemoryStore):
    def __init__(self):
        super().__init__()
        self.read_keys = []

    def get(self, key):
        value = super().get(key)
        self.read_keys.append(key)
        return value

logging_store = TLoggingStore()
logging_store.put('x', b'42')
assert logging_store.get('x') == b'42'
assert logging_store.read_keys == ['x']
try:
    logging_store.get('missing')
except KeyError:
    print('missing-key behavior preserved')

missing-key behavior preserved


### What we have proved

A wrapper is often **not** a subtype of the wrapped object. Use composition when the relationship is “uses/contains,” and single inheritance when a child can preserve the parent’s actual public contract.

---

## Problem 14 — Capstone: process heterogeneous events through one inherited API

**Challenge.** Construct a fully working event processor with a three-level branch, a sibling branch, validated inputs, template-method dispatch, and assertions. The engine may use polymorphism, but it must not inspect concrete event types.

**Before running anything:** Can one `process` method on a parent correctly handle grandchildren without `if type(event) ...`?

### Step 1 — Specify the contract before implementing

Every event has a validated identifier. The parent’s `process()` will call a hook `payload()` that descendants override. We deliberately separate common orchestration from variable behavior.

In [56]:
class TEvent:
    def __init__(self, event_id):
        if not isinstance(event_id, str) or not event_id.strip():
            raise ValueError('event_id must be nonempty text')
        self.event_id = event_id.strip()

    def payload(self):
        raise NotImplementedError('concrete event must supply payload')

    def process(self):
        return {'id': self.event_id, 'kind': type(self).__name__, 'payload': self.payload()}

### Step 2 — Create a numeric event that validates its own data

Finite numeric values are supported, but `bool` is intentionally excluded even though it subclasses `int`.

In [57]:
class TMetricEvent(TEvent):
    def __init__(self, event_id, name, value):
        super().__init__(event_id)
        if not isinstance(name, str) or not name.strip():
            raise ValueError('metric name must be nonempty')
        if isinstance(value, bool) or not isinstance(value, (int, float)) or not math.isfinite(value):
            raise ValueError('metric value must be finite and numeric')
        self.name = name.strip()
        self.value = float(value)

    def payload(self):
        return {'metric': self.name, 'value': self.value}

print(TMetricEvent('e-1', 'latency', 2.5).process())

{'id': 'e-1', 'kind': 'TMetricEvent', 'payload': {'metric': 'latency', 'value': 2.5}}


### Step 3 — Extend the metric branch with a grandchild

The threshold event inherits both parent validation and orchestration, then augments the payload. We call the parent’s `payload()` instead of duplicating its fields.

In [58]:
class TThresholdMetricEvent(TMetricEvent):
    def __init__(self, event_id, name, value, threshold):
        super().__init__(event_id, name, value)
        if isinstance(threshold, bool) or not isinstance(threshold, (int, float)) or not math.isfinite(threshold):
            raise ValueError('threshold must be finite and numeric')
        self.threshold = float(threshold)

    def payload(self):
        result = super().payload()
        return {**result, 'threshold': self.threshold, 'exceeded': self.value > self.threshold}

print(TThresholdMetricEvent('e-2', 'latency', 7, 5).process())

{'id': 'e-2', 'kind': 'TThresholdMetricEvent', 'payload': {'metric': 'latency', 'value': 7.0, 'threshold': 5.0, 'exceeded': True}}


### Step 4 — Create a sibling branch

A log event also inherits from `TEvent` but has a different payload. The processor will not need to know either concrete type.

In [59]:
class TLogEvent(TEvent):
    def __init__(self, event_id, message):
        super().__init__(event_id)
        if not isinstance(message, str) or not message.strip():
            raise ValueError('message must be nonempty text')
        self.message = message.strip()

    def payload(self):
        return {'message': self.message}

print(TLogEvent('e-3', 'started').process())

{'id': 'e-3', 'kind': 'TLogEvent', 'payload': {'message': 'started'}}


### Step 5 — Implement the generic engine

All events receive exactly the same `process()` call. Normal method dispatch chooses the correct payload implementation.

In [60]:
class TEventEngine:
    def __init__(self, events):
        self.events = tuple(events)
        if not all(isinstance(event, TEvent) for event in self.events):
            raise TypeError('every element must be a TEvent')

    def run(self):
        return [event.process() for event in self.events]

events = [
    TMetricEvent('e-1', 'latency', 2.5),
    TThresholdMetricEvent('e-2', 'latency', 7, 5),
    TLogEvent('e-3', 'started'),
]
report = TEventEngine(events).run()
for row in report:
    print(row)

{'id': 'e-1', 'kind': 'TMetricEvent', 'payload': {'metric': 'latency', 'value': 2.5}}
{'id': 'e-2', 'kind': 'TThresholdMetricEvent', 'payload': {'metric': 'latency', 'value': 7.0, 'threshold': 5.0, 'exceeded': True}}
{'id': 'e-3', 'kind': 'TLogEvent', 'payload': {'message': 'started'}}


### Step 6 — Verify identity, state, and branch-specific behavior

Tests should confirm relationships **and** outputs; one without the other does not establish correctness.

In [61]:
assert TThresholdMetricEvent.__bases__ == (TMetricEvent,)
assert issubclass(TThresholdMetricEvent, TEvent)
assert isinstance(events[1], TEvent)
assert report[0]['payload'] == {'metric': 'latency', 'value': 2.5}
assert report[1]['payload']['exceeded'] is True
assert report[2]['payload'] == {'message': 'started'}
print('capstone positive-path checks passed')

capstone positive-path checks passed


### Step 7 — Verify the rejection paths

Expected failures are caught so **Run All** still completes successfully. In real applications, choose a test framework such as `unittest` or `pytest` for a growing suite.

In [62]:
bad_cases = [
    lambda: TMetricEvent('m', 'cpu', float('nan')),
    lambda: TMetricEvent('m', 'cpu', True),
    lambda: TThresholdMetricEvent('t', 'cpu', 2, float('inf')),
    lambda: TLogEvent('l', '   '),
    lambda: TEventEngine([TEvent('ok'), object()]),
]
for index, build in enumerate(bad_cases, start=1):
    try:
        build()
    except (TypeError, ValueError):
        print('invalid case rejected:', index)
    else:
        raise AssertionError(f'invalid case {index} unexpectedly accepted')

invalid case rejected: 1
invalid case rejected: 2
invalid case rejected: 3
invalid case rejected: 4
invalid case rejected: 5


**Explanation.** `TEvent` is a concrete class with a `payload` stub here, so direct construction is possible but calling `process()` on it raises `NotImplementedError`. Problem 8 showed how an ABC can prevent such construction if your design requires an enforced abstract contract.

### What we have proved

This is a complete single-inheritance application: common event initialization at the root; a metric parent and threshold grandchild; a sibling log type; a shared processing template; and an engine with **zero concrete-class branches**.

---

## Problem 15 — Locate the class that actually defines an inherited method

**Challenge.** Build a reusable diagnostic function that identifies the first class in a single-inheritance MRO whose own dictionary defines an attribute. Contrast the result with the receiver’s runtime class.

**Before running anything:** If `leaf.method()` runs a method written three levels above, which class should a “method origin” diagnostic return?

### Step 1 — Define a class chain for inspection

The root defines `start`, the middle overrides `status`, and the leaf defines only `finish`.

In [63]:
class TJob:
    def start(self):
        return 'started'

    def status(self):
        return 'pending'

class TRunningJob(TJob):
    def status(self):
        return 'running'

class TFinishedJob(TRunningJob):
    def finish(self):
        return 'finished'

job = TFinishedJob()
print(job.start(), job.status(), job.finish())

started running finished


### Step 2 — Use the class MRO and each local dictionary

A normal attribute lookup uses the MRO. This helper specifically answers *which class first defines the name*; it is not a complete replacement for all descriptor or metaclass semantics.

In [64]:
def defining_class(cls, name):
    if not isinstance(cls, type):
        raise TypeError('cls must be a class')
    for ancestor in cls.__mro__:
        if name in ancestor.__dict__:
            return ancestor
    return None

for method_name in ('start', 'status', 'finish', 'missing'):
    origin = defining_class(TFinishedJob, method_name)
    print(method_name, '->', None if origin is None else origin.__name__)

start -> TJob
status -> TRunningJob
finish -> TFinishedJob
missing -> None


### Step 3 — Distinguish definition from object identity

The method’s location does not change the actual instance class. The runtime type of `job` is still the leaf class.

In [65]:
assert defining_class(TFinishedJob, 'start') is TJob
assert defining_class(TFinishedJob, 'status') is TRunningJob
assert defining_class(TFinishedJob, 'finish') is TFinishedJob
assert defining_class(TFinishedJob, 'missing') is None
assert type(job) is TFinishedJob
assert isinstance(job, TJob)
print('method origins and runtime types verified')

method origins and runtime types verified


### Step 4 — Use it as a debugging checklist

When inherited behavior is surprising, inspect the instance, its class, the MRO, the first class defining the name, and then any delegating `super()` calls.

In [66]:
print('runtime class:', type(job).__name__)
print('direct parent:', TFinishedJob.__bases__[0].__name__)
print('lookup path:', ' -> '.join(c.__name__ for c in TFinishedJob.__mro__))
print('status defined by:', defining_class(type(job), 'status').__name__)
print('status result:', job.status())

runtime class: TFinishedJob
direct parent: TRunningJob
lookup path: TFinishedJob -> TRunningJob -> TJob -> object
status defined by: TRunningJob
status result: running


### What we have proved

A method belongs to the class where it is *defined* but can execute on instances of many descendant classes. Local `__dict__` plus `__mro__` is a useful diagnostic for ordinary class methods in this single-inheritance tutorial.

---

## Final self-assessment: five synthesis problems

Try each without running earlier cells or scrolling to its solution. The answers are provided below as short, independently executable experiments.

1. **Prediction:** Explain why `isinstance(TCircle(), TShape)` is true but `type(TCircle()) is TShape` is false.
2. **Method location:** Predict the defining class for `TFinishedJob.status` and the runtime type of `TFinishedJob()`.
3. **Initialization:** What breaks if `TAssignedTicket.__init__` does not call `super()`?
4. **Special-method lookup:** Why did assigning to `positive.__len__` not affect `len(positive)`?
5. **Architecture:** Why does `TEventEngine` not need type checks for metric, threshold, and log events?

Read the concise answer cells below only after writing down your own responses.

### Answer 1 — compatible type versus exact type

`isinstance` accepts the class and its descendants; exact `type(...) is ...` excludes descendants. This is why polymorphic APIs commonly prefer `isinstance` or simply call a shared method.

In [67]:
sample = TCircle()
print(isinstance(sample, TShape), type(sample) is TShape)
assert isinstance(sample, TShape) and type(sample) is not TShape

True False


### Answer 2 — method origin versus runtime class

`TFinishedJob` inherits `status` from `TRunningJob`, but an object instantiated from the leaf is still a `TFinishedJob`.

In [68]:
print(defining_class(TFinishedJob, 'status').__name__, type(TFinishedJob()).__name__)

TRunningJob TFinishedJob


### Answer 3 — inherited-state invariants

Skipping `super().__init__` skips required parent initialization. The grandchild’s inherited `summary()` expects `reference` and `priority` fields established by its ancestors. The working chain delegates at each level.

In [69]:
print(TAssignedTicket('ticket-9', 1, 'Lee').summary())

ticket:ticket-9 priority:1 assignee:Lee


### Answer 4 — special method lookup

An explicit `obj.__len__()` uses ordinary attribute lookup, but implicit `len(obj)` looks for special behavior on the object’s type. Override `__len__` on a subclass to change the implicit behavior.

In [70]:
print('attribute call:', positive.__len__(), '| builtin call:', len(positive))

attribute call: 100 | builtin call: 2


### Answer 5 — polymorphism instead of branching

The engine invokes `event.process()` for each `TEvent`. The parent method invokes `self.payload()`, which dynamically dispatches to the concrete descendant implementation. This keeps the engine unaware of subclass details.

In [71]:
print([row['kind'] for row in TEventEngine(events).run()])

['TMetricEvent', 'TThresholdMetricEvent', 'TLogEvent']


---

## Further practice (deliberately unsolved)

These are extensions to the *new* worked problems, not recycled solutions from the earlier notebook.

- Change Problem 1 from category labels to **behavior** by giving each shape an inherited `describe()` method and overriding only when necessary; remove the entire classifier.
- Add an `area` comparison utility to Problem 4 that takes two `TEllipseGeometry` instances and never asks whether either is a circle.
- Add a `clear()` method to Problem 6’s `TGoodHistory`, then verify it never changes another object’s history.
- Add `__contains__` to Problem 7 so `x in collection` reflects its visible rather than hidden elements.
- Give Problem 10’s `TLabeledPoint` a specialized `parse_labeled('x,y,label')` factory while keeping its inherited two-coordinate `parse` intact.
- Extend Problem 14 with `TPercentageMetricEvent(TThresholdMetricEvent)` that enforces a value in `[0, 100]`, then verify the engine requires **no edits** to accept it.

**Study habit:** For every new hierarchy, write down the direct bases, predict at least three `isinstance`/`issubclass` outcomes, identify which class defines each method, and test both a successful and a failing path. If subclass validation narrows a base contract, explicitly decide whether inheritance is still appropriate.